In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature


from matplotlib.patches import Rectangle

import matplotlib.patheffects as path_effects

from copy import copy

import analysis_tools as tool
from config import *

# Load Data

In [ ]:
grid_data = tool.load_grid_data()

In [ ]:
dts = pd.date_range(start="2021-07-10T00", end="2021-07-15T21", freq="3h")

base_dir = "/automount/agh/s6tifohr/july21_eval/data"

exp_name = "REA"
da_rea = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp")
ds_rea_ens = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp")

exp_name = "BLK_CTL"
da_ctl = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp")
ds_ctl_ens = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp")

exp_name = "BLK_WLT"
da_wlt = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp")
ds_wlt_ens = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp")

exp_name = "BLK_SAT"
da_sat = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp")
ds_sat_ens = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp")

# Visualizations

## Precipitation Maps

In [ ]:
time_slice = slice(np.datetime64("2021-07-13T00"), np.datetime64("2021-07-15T12"))
timeframe = int((time_slice.stop - time_slice.start) / np.timedelta64(1, 'h'))

### Deterministic

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=3.2, h_pad=1.)

# Positions of subplots are required to align the colobars correctly:
pos_left = axs[1,0].get_position()  #(bottom) left subplot
pos_mid = axs[1,1].get_position()   #middle subplot
pos_right = axs[1,2].get_position() #subplot


# ICON-DREAM Reanalysis
ax = axs[0,0]
im = ax.tricontourf(grid_data["tri_26"], da_rea.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="ICON-DREAM")


# Saturation Run
ax = axs[0,1]
im = ax.tricontourf(grid_data["tri_26_red"], da_sat.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Wet Conditions")


# Wet - Ctrl
ax = axs[0,2]
mask_window = ((np.rad2deg(grid_data["grid_26_red"]["clon"]) >= PLOT_WINDOW[0]) & (np.rad2deg(grid_data["grid_26_red"]["clon"]) <= PLOT_WINDOW[1]) & 
               (np.rad2deg(grid_data["grid_26_red"]["clat"]) >= PLOT_WINDOW[2]) & (np.rad2deg(grid_data["grid_26_red"]["clat"]) <= PLOT_WINDOW[3]))

diff = da_sat.sel(step=time_slice).sum(dim="step") - da_ctl.sel(step=time_slice).sum(dim="step")
diff.data[~mask_window] = 0.

im = ax.tricontourf(grid_data["tri_26_red"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Wet - Control")


# Control Run
ax = axs[1,0]
im = ax.tricontourf(grid_data["tri_26_red"], da_ctl.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Control")


# Wilting Point Run
ax = axs[1,1]
im1 = ax.tricontourf(grid_data["tri_26_red"], da_wlt.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Dry Conditions")

# Add ax for left colorbar
cbar1_ax = fig.add_axes([pos_left.x1 + (pos_mid.x0 - pos_left.x1 - 1.5 * pos_left.width)/2, #left
                         pos_left.y0 - 0.1,     #bottom
                         pos_left.width * 1.5,  #width
                         0.02])                 #height
fig.colorbar(im1, cax=cbar1_ax, orientation='horizontal', label=f"{timeframe}h precipitation sum in mm")


# Dry - Ctrl
ax = axs[1,2]
diff = da_wlt.sel(step=time_slice).sum(dim="step") - da_ctl.sel(step=time_slice).sum(dim="step")
diff.data[~mask_window] = 0.

im2 = ax.tricontourf(grid_data["tri_26_red"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Dry - Control")


# Add colorbar for right column
cbar2_ax = fig.add_axes([pos_right.x0, pos_right.y0 - 0.1, pos_right.width, 0.02])
fig.colorbar(im2, cax=cbar2_ax, orientation='horizontal', label="Precipitation difference in mm")


for ax, char in zip(axs.reshape(-1), ["a", "b", "c", "d", "e", "f"]):
    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.05, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])
    
plt.savefig(f'./figs/map_comp_det.png', dpi=600, bbox_inches='tight', format='png')
plt.show()

### Ensemble

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=3.2, h_pad=1.)

# Positions of subplots are required to align the colobars correctly:
pos_left = axs[1,0].get_position()  #(bottom) left subplot
pos_mid = axs[1,1].get_position()   #middle subplot
pos_right = axs[1,2].get_position() #subplot


# ICON-DREAM Reanalysis
ax = axs[0,0]
im = ax.tricontourf(grid_data["tri_28"], ds_rea_ens.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="ICON-DREAM")


# Saturation Run
ax = axs[0,1]
im = ax.tricontourf(grid_data["tri_28"], ds_sat_ens.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Wet Conditions")


# Wet - Ctrl
ax = axs[0,2]
mask_window = ((np.rad2deg(grid_data["grid_28"]["clon"]) >= PLOT_WINDOW[0]) & (np.rad2deg(grid_data["grid_28"]["clon"]) <= PLOT_WINDOW[1]) & 
               (np.rad2deg(grid_data["grid_28"]["clat"]) >= PLOT_WINDOW[2]) & (np.rad2deg(grid_data["grid_28"]["clat"]) <= PLOT_WINDOW[3]))

diff = ds_sat_ens.sel(step=time_slice).sum(dim="step").median(dim="mem") - ds_ctl_ens.sel(step=time_slice).sum(dim="step").median(dim="mem")
diff.data[~mask_window] = 0.

im = ax.tricontourf(grid_data["tri_28"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Wet - Control")


# Control Run
ax = axs[1,0]
im = ax.tricontourf(grid_data["tri_28"], ds_ctl_ens.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Control")


# Wilting Point Run
ax = axs[1,1]
im1 = ax.tricontourf(grid_data["tri_28"], ds_wlt_ens.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Dry Conditions")

# Add ax for left colorbar
cbar1_ax = fig.add_axes([pos_left.x1 + (pos_mid.x0 - pos_left.x1 - 1.5 * pos_left.width)/2, #left
                         pos_left.y0 - 0.1,     #bottom
                         pos_left.width * 1.5,  #width
                         0.02])                 #height
fig.colorbar(im1, cax=cbar1_ax, orientation='horizontal', label=f"{timeframe}h precipitation sum in mm")


# Dry - Ctrl
ax = axs[1,2]
diff = ds_wlt_ens.sel(step=time_slice).sum(dim="step").median(dim="mem") - ds_ctl_ens.sel(step=time_slice).sum(dim="step").median(dim="mem")
diff.data[~mask_window] = 0.

im2 = ax.tricontourf(grid_data["tri_28"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Dry - Control")


# Add colorbar for right column
cbar2_ax = fig.add_axes([pos_right.x0, pos_right.y0 - 0.1, pos_right.width, 0.02])
fig.colorbar(im2, cax=cbar2_ax, orientation='horizontal', label="Precipitation difference in mm")


for ax, char in zip(axs.reshape(-1), ["a", "b", "c", "d", "e", "f"]):
    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.05, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.savefig(f'./figs/map_comp_ens.png', dpi=600, bbox_inches='tight', format='png')
plt.show()

## Time Series

### Deterministic

In [ ]:
# Compute time series of deterministic runs: 
cell_areas_focus = grid_data["area_26_red"].sel(cell=grid_data["focus_cells_26_red"])
ts_rea = (da_rea * grid_data["area_26"]).sel(cell=grid_data["focus_cells_26"]).sum(dim="cell")
ts_ctl = (da_ctl.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
ts_wlt = (da_wlt.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
ts_sat = (da_sat.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")

In [ ]:
fig, ax = plt.subplots()

ax.plot(ts_rea["valid_time"], ts_rea, color="black", label="REA")
ax.plot(ts_ctl["valid_time"], ts_ctl, color="tab:blue", label="CTL")
ax.plot(ts_wlt["valid_time"], ts_wlt, color="tab:green", label="WLT")
ax.plot(ts_sat["valid_time"], ts_sat, color="tab:orange", label="SAT")

plt.legend()

ax.set(ylabel="Area precipitation in kg/h")

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_deterministic.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

### Ensemble

In [ ]:
# Compute time series of ensemble runs:
cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
ts_rea_ens = (ds_rea_ens * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"]).sum(dim="cell")
ts_ctl_ens = (ds_ctl_ens.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
ts_wlt_ens = (ds_wlt_ens.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
ts_sat_ens = (ds_sat_ens.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")

In [ ]:
fig, ax = plt.subplots()

ax.plot(ts_ctl_ens["valid_time"], ts_ctl_ens.median(dim="mem"), color="tab:blue", label="CTL")
ax.fill_between(ts_ctl_ens["valid_time"], ts_ctl_ens.quantile(0.25, dim="mem"), ts_ctl_ens.quantile(0.75, dim="mem"), color="tab:blue", alpha=0.3)

ax.plot(ts_wlt_ens["valid_time"], ts_wlt_ens.median(dim="mem"), color="tab:green", label="WLT")
ax.fill_between(ts_wlt_ens["valid_time"], ts_wlt_ens.quantile(0.25, dim="mem"), ts_wlt_ens.quantile(0.75, dim="mem"), color="tab:green", alpha=0.3)

ax.plot(ts_sat_ens["valid_time"], ts_sat_ens.median(dim="mem"), color="tab:orange", label="SAT")
ax.fill_between(ts_sat_ens["valid_time"], ts_sat_ens.quantile(0.25, dim="mem"), ts_sat_ens.quantile(0.75, dim="mem"), color="tab:orange", alpha=0.3)

ax.plot(ts_rea_ens["valid_time"], ts_rea_ens.median(dim="mem"), color="black", label="REA")

plt.legend()

ax.set(ylabel="Area precipitation in kg/h")

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_ensemble.png', dpi=600, bbox_inches='tight', format='png')
plt.show()

## Animation

In [ ]:
from PIL import Image
import glob, os

In [ ]:
exp1 = "rea"
exp2 = "ctl"
varname = "TQV"

day_range = range(1,15)

lvls_anim = [0,10,20,30,40,50,60]
norm_anim = colors.BoundaryNorm(lvls_anim, 256)

fnames = {"rea": [f"data/moisture_tracking/icon_dream/det/icon_R03B07_{varname.lower()}_202107{dd:02}.nc" for dd in day_range],
          "ctl": [f"data/moisture_tracking/blcklst_ctl/det/icon_R03B07_{varname.lower()}_202107{dd:02}.nc" for dd in day_range],
          "wlt": [f"data/moisture_tracking/blcklst_wlt/det/icon_R03B07_{varname.lower()}_202107{dd:02}.nc" for dd in day_range],
          "sat": [f"data/moisture_tracking/blcklst_sat/det/icon_R03B07_{varname.lower()}_202107{dd:02}.nc" for dd in day_range]}

In [ ]:
ds1 = xr.open_mfdataset(fnames[exp1])[varname]
ds2 = xr.open_mfdataset(fnames[exp2])[varname]

old_frames = glob.glob("./figs/frames/*")
for f in old_frames:
    os.remove(f)

for frame in range(len(ds1.time)):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3), 
                                subplot_kw={'projection': ccrs.PlateCarree()})
    fig.tight_layout()

    # First subplot
    ax1.set_extent([-30, 60, 30, 65], crs=ccrs.PlateCarree())   # set map extent
    contourf = ax1.contourf(ds1["lon"], ds1["lat"], ds1.isel(time=frame), 
                            levels=lvls_anim, norm=norm_anim, cmap='viridis',
                            transform=ccrs.PlateCarree())
    ax1.set_title(exp1.upper())
    ax1.add_feature(cfeature.COASTLINE)

    # Box for region in which observations are passive
    box_passive_1 = Rectangle((-10, 29), 50, 45, edgecolor="white", linewidth=2, fill=False, alpha=0.7, transform=ccrs.PlateCarree())
    ax1.add_patch(box_passive_1)

    # Second subplot
    ax2.set_extent([-30, 60, 30, 65], crs=ccrs.PlateCarree())
    contourf = ax2.contourf(ds2["lon"], ds2["lat"], ds2.isel(time=frame), 
                            levels=lvls_anim, norm=norm_anim, cmap='viridis',
                            transform=ccrs.PlateCarree())
    ax2.set_title(exp2.upper())
    ax2.add_feature(cfeature.COASTLINE)

    box_passive_2 = Rectangle((-10, 29), 50, 45, edgecolor="white", linewidth=2, fill=False, alpha=0.7, transform=ccrs.PlateCarree())
    ax2.add_patch(box_passive_2)

    # Colorbar
    plt.subplots_adjust(bottom=0.01)
    pos = ax2.get_position()
    cbar_ax = fig.add_axes([pos.x0, 0, pos.width, 0.05])
    cbar = plt.colorbar(contourf, cax=cbar_ax, orientation="horizontal")
    fig.text(pos.x0-0.09, pos.y0-0.1, r"TQV in $kg/m^2$", weight="bold")

    fig.text(pos.x0-0.3, pos.y0-0.1, f"{str(ds2.isel(time=frame)["time"].values)[:13]}", color="black", weight="bold")

    plt.savefig(f"figs/frames/frame_{frame:04d}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


# Compile frames into animation:
frames = []
for filename in sorted(glob.glob('figs/frames/*.png')):
    frames.append(Image.open(filename))

frames[0].save(f"./figs/animation_{varname}.gif", save_all=True, append_images=frames[1:], 
               duration=200, loop=0)

# Assimilation Region Showcase

In [ ]:
# This is not actually coinciding with the passive region, which is -10 to 40E and 30 to 75N, 
# but I used it to get a background in the image below  
mask_passive_region = ((np.rad2deg(grid_data["grid_det"]["clon"]) >= -20) & (np.rad2deg(grid_data["grid_det"]["clon"]) <= 50) & 
                       (np.rad2deg(grid_data["grid_det"]["clat"]) >= 20) & (np.rad2deg(grid_data["grid_det"]["clat"]) <= 85)).values
lons_pr, lats_pr = np.rad2deg(grid_data["grid_det"]["clon"][mask_passive_region]), np.rad2deg(grid_data["grid_det"]["clat"][mask_passive_region])

ii_passive_region = np.arange(len(mask_passive_region))[mask_passive_region]

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent([-30, 70, 25, 80], crs=ccrs.PlateCarree())

gl1 = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, alpha=0.15, zorder=10, color="black")
gl1.top_labels = False
gl1.right_labels = False

im = ax.tricontourf(grid_data["tri_28"], np.zeros(len(grid_data["area_28"])))
ax.text(50, 32, "EU Nest", color="white", weight="bold")

# Create a rectangle patch for the passive observation region:
box_passive = Rectangle((PASSIVE_REGION["x0"], PASSIVE_REGION["y0"]), PASSIVE_REGION["wx"], PASSIVE_REGION["wy"], edgecolor="purple", linewidth=2, fill=False)
ax.add_patch(box_passive)
ax.text(-9, 72, "Passive Observations", color="purple", weight="bold")

# Create a rectangle patch for the focus region
box_focus = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor="orange", linewidth=2, fill=False)
ax.add_patch(box_focus)
ax.text(11, 50, "Focus Region", color="orange", weight="bold")

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)

plt.savefig("figs/demo_nest_passive_obs.png", dpi=600, bbox_inches='tight', format='png')
plt.show()